In [23]:
import pandas as pd
import networkx as nx
import numpy as np
from collections import defaultdict
from networkx.algorithms import isomorphism as iso

In [24]:
def load_graph(path):
    df = pd.read_csv(path)
    src, dst = df.columns[:2]

    G = nx.DiGraph()
    G.add_edges_from(zip(df[src], df[dst]))
    return G

MAOL = load_graph(r"C:\Users\vachi\Downloads\maol_1.1_edge_list.csv")
FAFB = load_graph(r"C:\Users\vachi\Downloads\fafb_783_edge_list.csv")
BANC = load_graph(r"C:\Users\vachi\Downloads\banc_626_edge_list (1).csv")

In [25]:
def fingerprint(G):
    fp = {}

    for n in G.nodes():
        fp[n] = (
            G.in_degree(n),
            G.out_degree(n),
            G.in_degree(n) + G.out_degree(n)
        )

    return fp

FP = {
    "MAOL": fingerprint(MAOL),
    "FAFB": fingerprint(FAFB),
    "BANC": fingerprint(BANC)
}

In [30]:
def bin_key(fp_tuple):
    # coarse rounding = allows biological variation
    return (
        round(fp_tuple[0] / 2),
        round(fp_tuple[1] / 2),
        round(fp_tuple[2] / 2)
    )

binned = {
    name: defaultdict(list)
    for name in FP
}

for name in FP:
    for node, f in FP[name].items():
        binned[name][bin_key(f)].append(node)

In [31]:
common_keys = (
    set(binned["MAOL"].keys())
    & set(binned["FAFB"].keys())
    & set(binned["BANC"].keys())
)

triplets = []

for k in common_keys:
    A_nodes = binned["MAOL"][k][:5]
    B_nodes = binned["FAFB"][k][:5]
    C_nodes = binned["BANC"][k][:5]

    for a in A_nodes:
        for b in B_nodes:
            for c in C_nodes:
                triplets.append((a, b, c))

print("candidate triplets:", len(triplets))

candidate triplets: 154943


In [33]:
def expand_local(G, node, depth=1):
    nodes = set([node])
    frontier = [node]

    for _ in range(depth):
        new = []
        for n in frontier:
            new.extend(list(G.successors(n))[:2])
        nodes.update(new)
        frontier = new

    return G.subgraph(nodes).copy()

In [34]:
valid = []

for a, b, c in triplets:

    GA = expand_local(MAOL, a, depth=1)
    GB = expand_local(FAFB, b, depth=1)
    GC = expand_local(BANC, c, depth=1)

    # quick pruning first
    if abs(len(GA) - len(GB)) > 3 or abs(len(GA) - len(GC)) > 3:
        continue

    GM = iso.DiGraphMatcher(GA, GB)

    if GM.is_isomorphic():
        GM2 = iso.DiGraphMatcher(GA, GC)
        if GM2.is_isomorphic():
            valid.append((a, b, c))

print("valid circuits:", len(valid))

valid circuits: 14508


In [35]:
def expand_final(G, nodes):
    expanded = set(nodes)
    for n in nodes:
        expanded.update(list(G.successors(n))[:2])
    return expanded

In [36]:
best = None
best_size = 0

for a, b, c in valid:

    A = expand_final(MAOL, [a])
    B = expand_final(FAFB, [b])
    C = expand_final(BANC, [c])

    size = min(len(A), len(B), len(C))

    if size > best_size:
        best_size = size
        best = (A, B, C)

print("best size:", best_size)

best size: 3


In [37]:
A, B, C = best

N = min(len(A), len(B), len(C))

df = pd.DataFrame({
    "MAOL": list(A)[:N],
    "FAFB": list(B)[:N],
    "BANC": list(C)[:N]
})

df.to_csv("network.csv", index=False)

df.head()

,MAOL,FAFB,BANC
0,28699,720575940603925280,720575941624757642
1,94154,720575940618369659,720575941436390188
2,40107,720575940638114357,720575941595323687
